# Notebook 08 — Trajectory Analysis and Pseudotime (Survey)

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence | **This notebook: Tier 3 Survey**  
**Notebook:** 08 of 12  
**Time estimate:** 60 minutes

> Cells don't just sit in discrete states — they transition between states.
> Trajectory analysis orders cells along a continuous developmental path
> and assigns each cell a "pseudotime" coordinate.

---
## Step 1 — Motivation

Clustering identifies discrete cell types. But many biological processes are continuous:
stem cell differentiation, immune cell activation, cell cycle progression. In these
cases, cells captured by scRNA-seq represent different points along a trajectory, not
distinct types. Pseudotime analysis reveals this continuous structure.

---
## Step 2 — Intuition

**The key insight:** If you have a snapshot of a differentiating cell population,
the cells near the start of differentiation have different expression profiles from
cells near the end. By ordering cells in expression space, you can reconstruct the
temporal progression even without time-series data.

**Pseudotime:** A scalar value assigned to each cell representing its position along
a trajectory. Pseudotime 0 = start; pseudotime 1 = end.

**Minimum spanning tree (MST) approach (Monocle 1):**  
Build a graph where nodes are cells and edges are weighted by expression distance.
The MST connects all cells with minimum total edge weight. The longest path through
the MST is the main trajectory axis.

---
## Step 3 — Biological Background

**Major trajectory analysis tools:**

| Tool | Year | Approach | Strengths |
|------|------|----------|-----------|
| Monocle 1 | 2014 | MST on ICA | First pseudotime method |
| DPT | 2016 | Diffusion pseudotime | Robust to noise |
| Monocle 3 | 2019 | UMAP + MST | Branching, scalable |
| RNA velocity | 2018 | Spliced/unspliced ratio | Directionality from data |
| PAGA | 2019 | Graph abstraction | Partition-based |

**Diffusion Pseudotime (DPT — Haghverdi 2016):**  
Models cell-to-cell transitions as a random walk (diffusion process) on the kNN graph.
The pseudotime of a cell is the expected number of steps a random walk from the root
cell takes to reach it. Implemented in scanpy as `sc.tl.dpt()`.

**RNA Velocity (La Manno 2018):**  
Uses the ratio of unspliced (nascent) to spliced (mature) mRNA to predict the
future direction of each cell's expression state. Unlike pseudotime, RNA velocity
gives directionality without choosing a root cell.

---
## Step 4 — Mathematical Explanation

**Diffusion map / Diffusion pseudotime:**

1. Build kernel matrix $K_{ij} = \exp(-\|x_i - x_j\|^2 / \epsilon)$ from PCA distances
2. Row-normalize to get Markov transition matrix $M_{ij} = K_{ij} / \sum_j K_{ij}$
3. Eigendecompose $M = \Phi \Lambda \Phi^{-1}$
4. Diffusion coordinates: $\text{DC}_k(i) = \lambda_k^t \phi_k(i)$ (k-th eigenvector,
   scaled by $k$-th eigenvalue to the power $t$, the diffusion time)
5. Pseudotime from root cell $r$:
   $\text{DPT}(i) = \|\phi(i) - \phi(r)\|_{\text{diffusion}}$

The first non-trivial diffusion component separates cells by their trajectory
position.

In [ ]:
# Step 6 — Python: Minimum spanning tree pseudotime from scratch
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

rng = np.random.default_rng(42)

# ---- Simulate a differentiation trajectory ----
# Progenitor → Intermediate → Mature (with noise)
N = 200
t_true = rng.uniform(0, 1, N)  # true developmental time
N_DIMS = 20

# Expression changes linearly along the trajectory with noise
X_traj = np.zeros((N, N_DIMS))
for d in range(N_DIMS):
    slope = rng.normal(0, 2.0)  # some genes increase, some decrease
    X_traj[:, d] = slope * t_true + rng.normal(0, 0.4, N)

# Add a branch (last 30% split into two fates)
branch_cells = t_true > 0.65
branch_fate = rng.choice([0, 1], N)
for i in np.where(branch_cells)[0]:
    if branch_fate[i] == 1:
        X_traj[i, :5] += 3.0  # Fate B activates different genes

# ---- Compute PCA (2D for display) ----
Xc = X_traj - X_traj.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
Z = (U * s)[:, :2]

# ---- MST-based pseudotime ----
def mst_from_matrix(D):
    """Prim's algorithm: minimum spanning tree on full distance matrix."""
    n = D.shape[0]
    in_tree = np.zeros(n, dtype=bool)
    min_dist = np.full(n, np.inf)
    parent = np.full(n, -1, dtype=int)
    min_dist[0] = 0
    edges = []
    for _ in range(n):
        # Find minimum distance node not yet in tree
        candidates = np.where(~in_tree)[0]
        i = candidates[np.argmin(min_dist[candidates])]
        in_tree[i] = True
        if parent[i] >= 0:
            edges.append((parent[i], i))
        # Update distances
        for j in range(n):
            if not in_tree[j] and D[i, j] < min_dist[j]:
                min_dist[j] = D[i, j]
                parent[j] = i
    return edges

# Use PCA coordinates for MST
D = cdist(Z, Z)
print('Building MST...')
mst_edges = mst_from_matrix(D)
print(f'MST: {len(mst_edges)} edges')

# ---- Pseudotime: BFS from root cell ----
# Root = cell with smallest t_true (most progenitor-like)
root = np.argmin(t_true)
from collections import defaultdict, deque
graph = defaultdict(list)
for u, v in mst_edges:
    graph[u].append(v)
    graph[v].append(u)

# BFS to get pseudotime (graph distance from root)
pseudotime = np.full(N, -1.0)
pseudotime[root] = 0.0
queue = deque([root])
visited = {root}
while queue:
    node = queue.popleft()
    for nbr in graph[node]:
        if nbr not in visited:
            visited.add(nbr)
            pseudotime[nbr] = pseudotime[node] + D[node, nbr]
            queue.append(nbr)
# Normalize pseudotime to [0, 1]
pseudotime = (pseudotime - pseudotime.min()) / (pseudotime.max() - pseudotime.min())

corr = np.corrcoef(t_true, pseudotime)[0, 1]
print(f'Pearson correlation with true time: {corr:.3f}')

In [ ]:
# Step 7 — Visualization: Trajectory and pseudotime
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: True developmental time
sc0 = axes[0].scatter(Z[:, 0], Z[:, 1], c=t_true, cmap='viridis', s=20, alpha=0.8)
plt.colorbar(sc0, ax=axes[0], label='True developmental time')
axes[0].set_title('A. True developmental time\n(ground truth)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Panel B: MST edges + pseudotime
for u, v in mst_edges:
    axes[1].plot([Z[u, 0], Z[v, 0]], [Z[u, 1], Z[v, 1]],
                  'k-', lw=0.3, alpha=0.3, zorder=0)
sc1 = axes[1].scatter(Z[:, 0], Z[:, 1], c=pseudotime, cmap='plasma', s=15, alpha=0.8, zorder=1)
plt.colorbar(sc1, ax=axes[1], label='Pseudotime (MST)')
axes[1].scatter(Z[root, 0], Z[root, 1], c='red', s=100, marker='*',
                  zorder=2, label='root')
axes[1].legend(fontsize=8)
axes[1].set_title(f'B. MST pseudotime\n(r={corr:.2f} vs. true time)')
axes[1].set_xlabel('PC1')

# Panel C: Pseudotime vs. true time scatter
axes[2].scatter(t_true, pseudotime, s=10, alpha=0.5, c=t_true, cmap='viridis')
axes[2].set_xlabel('True developmental time')
axes[2].set_ylabel('Pseudotime (MST)')
axes[2].set_title(f'C. Pseudotime vs. true time\nPearson r = {corr:.3f}')

plt.suptitle('Module 10 NB08: Trajectory Analysis — Survey', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Change the root cell to one with high `t_true`. How does the pseudotime correlation
   change? What does this tell you about root cell selection?
2. What happens to the MST when a bifurcation (branching point) exists? Can MST
   pseudotime handle this? What algorithm would you use instead?
3. Look up RNA velocity. In one paragraph: what are "spliced" and "unspliced" mRNA
   and how does their ratio indicate the direction of cell state change?

---
## Step 10 — Quiz

1. What is pseudotime? What is it NOT (i.e., what can it not tell you)?
2. Name two pseudotime algorithms and their key difference.
3. What is RNA velocity and why doesn't it need a root cell?
4. What is the difference between a linear trajectory and a branching trajectory?
5. Why can't you use static clustering (Leiden) to study continuous differentiation?

---
## Step 12 — Reflection

> *[In one paragraph: describe a biological scenario where pseudotime analysis
> would be more informative than standard clustering, and explain why.]*

---
*Next: `09_mass_spectrometry_proteomics.ipynb`*